# PETalytics — Stage 1: ESM-2 Embedding Generation

**What this notebook does:** loads sequences from `petadex/catalytic-orfs-90pid` (the 18.17M-sequence
catalytic-ORF corpus on Hugging Face), cleans/subsamples them, runs them through **ESM-2** , and saves mean-pooled sequence embeddings to disk.


## 1. Install dependencies

In [ ]:
!pip install -q transformers accelerate datasets pyarrow huggingface_hub

In [ ]:
!nvidia-smi

Thu Jul 16 03:31:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             53W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Imports and device check

In [ ]:
import os
import time
import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, EsmModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: no GPU detected. Runtime > Change runtime type > select a GPU before continuing.')

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
GPU memory (GB): 85.1


## 3. Mount Google Drive (for persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Verify the mount actually worked and points at your real Drive, not a failed/empty mount.
!ls /content/drive/MyDrive | head -5
os.makedirs(OUTPUT_DIR if 'OUTPUT_DIR' in dir() else '/content/drive/MyDrive/petalytics/embeddings', exist_ok=True)
print('If the folder listing above looks like your real Drive (not empty/an error), the mount is good.')
print('After section 7 finishes, check drive.google.com directly in another tab as the most trustworthy',
      'confirmation that files actually landed there.')

2015_science_amstad.pdf
20181105_163956_HDR.jpg
2020-04-30 08.44.03.pdf
20250825_181451.jpg
2336
If the folder listing above looks like your real Drive (not empty/an error), the mount is good.
After section 7 finishes, check drive.google.com directly in another tab as the most trustworthy confirmation that files actually landed there.


## 4. Config

In [ ]:
SUBSAMPLE_N   = 50000        # bumped from 20k given the extra compute headroom. Full corpus is
                               # ~18.17M rows — still a small fraction, room to go further if fast.
MAX_SEQ_LEN   = 1024           # truncate outlier-long sequences (corpus ranges 30 to ~59,300 residues;
                               # this just guards against rare extreme outliers blowing up memory).
BATCH_SIZE    = 64             # bumped from 32. If you got an A100/L4 and see low GPU memory usage
                               # in `!nvidia-smi`, try 128. If you land on a T4 and hit CUDA OOM, drop
                               # back to 32.
MODEL_NAME    = 'facebook/esm2_t30_150M_UR50D'   # stepped up from the 35M free-tier default (640-dim
                               # vs 480-dim) — richer embeddings, still fast on any Colab GPU tier.
                               # If throughput is great after the first checkpoint, esm2_t33_650M_UR50D
                               # (1280-dim, the size most PETase/pLM papers actually use) is worth
                               # trying given how much runway is left.
RANDOM_SEED   = 42
OUTPUT_DIR    = '/content/drive/MyDrive/petalytics/embeddings'
CHECKPOINT_EVERY = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## 5. Load, clean, and subsample the dataset
**Pinned to a specific dataset revision** (see cell below) — this matters because PETadex is actively
being reworked, and `ORF ID` values aren't guaranteed stable across revisions. Without pinning, a
disconnect-and-resume hours later could silently re-resolve to an updated revision where an old
`ORF ID` now points at a *different* sequence — the resume logic would see 'ID already done' and skip
it, keeping a stale embedding under the wrong sequence's identity. Pinning removes that risk for
today's run, and the revision hash gets saved to Drive so a resume after a full restart uses the same
frozen snapshot rather than whatever's current at that moment.

Same cleaning logic as before: flags reference proteins (`Genbank Acc` populated) vs. predicted ORFs
(`Library`/`ORF Start`/`ORF End` populated, may be fragments), filters to standard amino acids, filters
by length, then takes a random subsample.

In [ ]:
from huggingface_hub import HfApi

revision_pin_path = os.path.join(OUTPUT_DIR, 'dataset_revision.txt')
if os.path.exists(revision_pin_path):
    with open(revision_pin_path) as f:
        DATASET_REVISION = f.read().strip()
    print('Resuming with previously pinned revision:', DATASET_REVISION)
else:
    DATASET_REVISION = HfApi().dataset_info('petadex/catalytic-orfs-90pid').sha
    with open(revision_pin_path, 'w') as f:
        f.write(DATASET_REVISION)
    print('Pinned new dataset revision (saved to Drive for future resumes):', DATASET_REVISION)

ds = load_dataset('petadex/catalytic-orfs-90pid', split='train', revision=DATASET_REVISION)
print(ds)
print(ds.column_names)

STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWYX')

def is_clean(seq):
    if seq is None or len(seq) == 0:
        return False
    return set(seq.upper()).issubset(STANDARD_AA)

def add_provenance(example):
    example['provenance'] = 'reference' if example.get('Genbank Acc') else 'predicted_orf'
    return example

# Shuffle + take an oversampled candidate pool FIRST, before running any per-row Python function on
# the full 17.8M-row train split. map()/filter() invoke a Python callback per row, so running them on
# the full corpus first is needlessly slow (~30+ min at ~10k rows/sec) when we only want SUBSAMPLE_N
# rows at the end. Slicing down to a manageable pool first, then only mapping/filtering that pool,
# cuts this from ~17.8M row-operations down to roughly OVERSAMPLE_FACTOR * SUBSAMPLE_N.
OVERSAMPLE_FACTOR = 2  # pool is 30% bigger than SUBSAMPLE_N, to leave margin for rows dropped by
                          # the cleanliness/length filter below.
candidate_pool_size = min(int(SUBSAMPLE_N * OVERSAMPLE_FACTOR), len(ds))
ds = ds.shuffle(seed=RANDOM_SEED).select(range(candidate_pool_size))
print(f'Candidate pool after shuffle+select: {len(ds)} rows (before cleaning)')

ds = ds.map(add_provenance)
ds = ds.filter(lambda ex: is_clean(ex['Sequence']) and len(ex['Sequence']) <= MAX_SEQ_LEN)
print('Remaining after cleaning/length filter:', len(ds))

if len(ds) < SUBSAMPLE_N:
    print(f'WARNING: only {len(ds)} rows survived filtering, short of the target {SUBSAMPLE_N}. '
          f'Increase OVERSAMPLE_FACTOR above and re-run this cell.')

ds = ds.select(range(min(SUBSAMPLE_N, len(ds))))
print('Final subsample size:', len(ds))
print('Provenance breakdown:', {p: ds['provenance'].count(p) for p in set(ds['provenance'])})

Resuming with previously pinned revision: 1578cb113d6b08489136b6dab98c667411d7ec19
Dataset({
    features: ['ORF ID', 'Genbank Acc', 'Library', 'Contig', 'ORF Start', 'ORF End', 'ORF Type', 'Sequence'],
    num_rows: 17809500
})
['ORF ID', 'Genbank Acc', 'Library', 'Contig', 'ORF Start', 'ORF End', 'ORF Type', 'Sequence']
Candidate pool after shuffle+select: 100000 rows (before cleaning)


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100000 [00:00<?, ? examples/s]

Remaining after cleaning/length filter: 56356
Final subsample size: 50000
Provenance breakdown: {'reference': 8443, 'predicted_orf': 41557}


## 6. Load ESM-2 and sanity-check output shape
`EsmModel` is the bare encoder (no masked-LM head) — the right choice for extracting embeddings.
`last_hidden_state` is the per-residue output directly; no need to dig through `hidden_states`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME).to(device).eval()
print('Hidden size:', model.config.hidden_size)

_test_seq = ds[0]['Sequence']
_inputs = tokenizer(_test_seq, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(device)
with torch.no_grad():
    _out = model(**_inputs)
print('last_hidden_state shape:', _out.last_hidden_state.shape)  # expect (1, seq_len+2, hidden_size)
# Note the +2: ESM-2's tokenizer adds a BOS and EOS token. The mean-pooling step below uses the
# attention mask, so these don't need special-casing, but they DO get included in the mean unless
# you want to exclude them — flag this to me if you'd rather pool over residues only.

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/595M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/486 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hidden size: 640
last_hidden_state shape: torch.Size([1, 100, 640])


## 7. Embedding extraction (batched, mean-pooled, checkpointed)
Resumable: re-running this cell after a disconnect skips sequences already embedded.

In [ ]:
embeddings_path = os.path.join(OUTPUT_DIR, 'esm2_150M_embeddings.npy')
ids_path = os.path.join(OUTPUT_DIR, 'esm2_150M_ids.npy')

all_ids = np.array(ds['ORF ID'])
sequences = ds['Sequence']

if os.path.exists(embeddings_path) and os.path.exists(ids_path):
    done_embeddings = list(np.load(embeddings_path))
    done_ids = list(np.load(ids_path))
    done_id_set = set(done_ids)
    print(f'Resuming: {len(done_ids)} sequences already embedded.')
else:
    done_embeddings, done_ids, done_id_set = [], [], set()

remaining_idx = [i for i, sid in enumerate(all_ids) if sid not in done_id_set]
print(f'{len(remaining_idx)} sequences left to embed out of {len(all_ids)}.')

start_time = time.time()
for batch_num, start in enumerate(range(0, len(remaining_idx), BATCH_SIZE)):
    batch_idx = remaining_idx[start:start + BATCH_SIZE]
    batch_seqs = [sequences[i] for i in batch_idx]
    batch_ids = [all_ids[i] for i in batch_idx]

    inputs = tokenizer(batch_seqs, return_tensors='pt', padding=True, truncation=True,
                        max_length=MAX_SEQ_LEN).to(device)
    with torch.no_grad():
        out = model(**inputs)
    last_hidden = out.last_hidden_state                        # (batch, seq_len, hidden)
    mask = inputs['attention_mask'].unsqueeze(-1).float()      # (batch, seq_len, 1)
    summed = (last_hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    mean_pooled = (summed / counts).cpu().numpy()               # (batch, hidden)

    done_embeddings.extend(mean_pooled)
    done_ids.extend(batch_ids)

    if batch_num % CHECKPOINT_EVERY == 0:
        np.save(embeddings_path, np.array(done_embeddings))
        np.save(ids_path, np.array(done_ids))
        elapsed = time.time() - start_time
        rate = len(done_ids) / elapsed if elapsed > 0 else 0
        print(f'Checkpoint: {len(done_ids)}/{len(all_ids)} done '
              f'({rate:.1f} seq/sec, {elapsed:.0f}s elapsed)')

np.save(embeddings_path, np.array(done_embeddings))
np.save(ids_path, np.array(done_ids))
print('Done. Final shape:', np.array(done_embeddings).shape)

50000 sequences left to embed out of 50000.
Checkpoint: 64/50000 done (47.5 seq/sec, 1s elapsed)
Checkpoint: 3264/50000 done (49.9 seq/sec, 65s elapsed)
Checkpoint: 6464/50000 done (50.0 seq/sec, 129s elapsed)
Checkpoint: 9664/50000 done (50.2 seq/sec, 192s elapsed)
Checkpoint: 12864/50000 done (51.0 seq/sec, 252s elapsed)
Checkpoint: 16064/50000 done (51.2 seq/sec, 314s elapsed)
Checkpoint: 19264/50000 done (51.1 seq/sec, 377s elapsed)
Checkpoint: 22464/50000 done (50.8 seq/sec, 442s elapsed)
Checkpoint: 25664/50000 done (50.6 seq/sec, 507s elapsed)
Checkpoint: 28864/50000 done (50.5 seq/sec, 571s elapsed)
Checkpoint: 32064/50000 done (50.6 seq/sec, 634s elapsed)
Checkpoint: 35264/50000 done (50.6 seq/sec, 697s elapsed)
Checkpoint: 38464/50000 done (50.4 seq/sec, 763s elapsed)
Checkpoint: 41664/50000 done (50.4 seq/sec, 826s elapsed)
Checkpoint: 44864/50000 done (50.5 seq/sec, 888s elapsed)
Checkpoint: 48064/50000 done (50.6 seq/sec, 950s elapsed)
Done. Final shape: (50000, 640)


## 8. Sanity checks

In [ ]:
embs = np.load(embeddings_path)
ids = np.load(ids_path)
print('Embeddings shape:', embs.shape)
print('Any NaNs?', np.isnan(embs).any())
print('Any all-zero rows?', (embs == 0).all(axis=1).sum())
print('Per-dimension std (first 10 dims):', embs.std(axis=0)[:10])

Embeddings shape: (50000, 640)
Any NaNs? False
Any all-zero rows? 0
Per-dimension std (first 10 dims): [0.05242443 0.05261268 0.09514524 0.05056794 0.06364802 0.04961554
 0.04740227 0.05614566 0.04925724 0.07717896]
